In [1]:
# Check for GPU
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


***Required Libraries and Environment Setup***

In [2]:
# Import essential libraries for data processing, visualization, and deep learning
import numpy as np               # Array operations and numerical computations
import pandas as pd              # Data manipulation and analysis
import matplotlib.pyplot as plt  # Data visualization
import seaborn as sns            # Advanced data visualization
import torch                     # PyTorch deep learning framework
import torch.nn as nn            # Neural network modules
from torch.utils.data import DataLoader, TensorDataset  # Data loading utilities
from tqdm import tqdm            # Progress bar for loops
from collections import Counter  # Frequency counting
from string import punctuation   # Punctuation character set
import re                        # Python Regex

In [3]:
# Import additional libraries
import time

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


**Data Exploration**

The IMDB Movie Review Dataset is a widely-used benchmark for sentiment analysis, containing 50,000 highly polar movie reviews labeled as either positive (1) or negative (0). Each review is represented as raw text and requires extensive preprocessing before model training.

Initial exploration of the dataset provides critical insights into its structure, including the distribution of classes and the characteristics of the text data. These insights inform subsequent preprocessing decisions and model design choices.

In [5]:
DATASET_FILENAME = 'IMDB Dataset 5K.csv'

# Load the dataset into a pandas DataFrame for analysis
df = pd.read_csv(DATASET_FILENAME)

Text normalization is a critical preprocessing step that standardizes the format of the text data, reducing the dimensionality and noise in the input space. Common normalization techniques include:

Case Normalization: Converting all text to lowercase to treat words like "Good" and "good" as the same token
Removing th html tags: Usually most of these textss are extracted from website source code(html code), hence we see texts like <br>, <br />, etc because they are html codes interleaved with the actual texts. These entities has to be removed to get a clean sentence.
Punctuation Removal: Eliminating punctuation marks that generally don't carry sentiment information
Special Character Handling: Removing or replacing special characters and symbols
These preprocessing steps help create a more standardized representation of the text, improving the generalization capabilities of the model.

In [6]:
# Examine class distribution
sentiment_counts = df['sentiment'].value_counts()
# Analyze review length statistics
df['review_length'] = df['review'].apply(len)
# Step 1: Case normalization - Convert all text to lowercase
df['review_normalized'] = df['review'].apply(lambda x: x.lower())
original_text = df['review'].iloc[1][:100]
normalized_text = df['review_normalized'].iloc[1][:100]
# Step 2: Remove HTML tags
df['review_no_html'] = df['review_normalized'].apply(
    lambda x: re.sub(r'<.*?>', '', x)
)
# Step 3: Punctuation removal
df['clean_text'] = df['review_no_html'].apply(lambda x: ''.join(' ' if c in punctuation else c for c in x))
# Step 4: Calculate text length after preprocessing
df['clean_text_length'] = df['clean_text'].apply(len)
display(df[['sentiment', 'review_length', 'clean_text_length']].head())
length_reduction = (df['review_length'].mean() - df['clean_text_length'].mean()) / df['review_length'].mean() * 100

,sentiment,review_length,clean_text_length
0,positive,1761,1725
1,positive,998,962
2,positive,926,902
3,negative,748,712
4,positive,1317,1269


Text Tokenization and Encoding

Vocabulary Construction

Tokenization involves breaking down text into smaller units (tokens), which are then mapped to numerical indices. This process creates a vocabulary that serves as the foundation for word embeddings.

The steps involved in vocabulary construction include:

Splitting text into individual words
Counting word frequencies to identify common terms
Creating a word-to-index mapping for numerical representation
Handling special tokens (e.g., padding, unknown words)
This approach transforms raw text into structured numerical data suitable for neural network processing.

In [7]:
# Step 1: Create a corpus of all cleaned text for vocabulary construction
all_text = ' '.join(df['clean_text'].tolist())
# Step 2: Split corpus into individual words
words = all_text.split()
# Step 3: Count word frequencies
word_counts = Counter(words)
total_unique_words = len(word_counts)
# Step 4: Create word-to-index mapping (vocabulary)
# Sort words by frequency (most common first) to optimize index assignments
sorted_words = word_counts.most_common(total_unique_words)

# Create word-to-index dictionary
# We add +1 to all indices to reserve index 0 for padding
vocab_to_int = {word: idx+1 for idx, (word, _) in enumerate(sorted_words)}

Review Encoding and Label Processing
After constructing the vocabulary, we need to:

Convert each review from words to integer sequences using the vocabulary mapping
Transform categorical sentiment labels into numerical values (binary classification)
These steps complete the data transformation process, preparing the dataset for neural network training.

In [8]:
# Step 1: Prepare reviews for encoding
reviews_split = df['clean_text'].tolist()

# Step 2: Encode reviews as integer sequences
reviews_encoded = []

for review in reviews_split:
    # Convert each word in the review to its corresponding integer index
    encoded_review = [vocab_to_int[word] for word in review.split() if word in vocab_to_int]
    reviews_encoded.append(encoded_review)

# Display statistics about encoded reviews
review_lengths = [len(review) for review in reviews_encoded]

In [9]:
# Step 3: Encode sentiment labels
sentiment_categories = df['sentiment'].tolist()

# Convert sentiment categories to binary labels (1 for positive, 0 for negative)
encoded_labels = np.array([1 if label == 'positive' else 0 for label in sentiment_categories])

Dataset Preparation for Neural Networks
Sequence Standardization
Neural networks typically require fixed-size inputs, but our encoded reviews have variable lengths. To address this challenge, we need to standardize all sequences to the same length through:

Padding: Adding zeros to shorter sequences
Truncation: Shortening excessively long sequences
This process enables batch processing and proper tensor operations in the neural network architecture. The sequence length is an important hyperparameter that balances information retention with computational efficiency.

In [10]:
# Define a function to standardize sequence lengths through padding and truncation
def pad_features(reviews_int, seq_length):
    """
    Return features of review_ints, where each review is padded with zeros or
    truncated to the input seq_length.

    Parameters:
    -----------
    reviews_int : List[List[int]]
        List of reviews where each review is a list of integer word indices
    seq_length : int
        Fixed sequence length for all reviews

    Returns:
    --------
    features : np.ndarray
        Array of padded/truncated integer sequences with shape (len(reviews_int), seq_length)
    """
    # Initialize an array of zeros with shape (n_reviews, seq_length)
    features = np.zeros((len(reviews_int), seq_length), dtype=int)

    # Iterate through reviews
    for i, review in enumerate(reviews_int):
        review_len = len(review)
        # Handle short sequences: add padding zeros at the beginning
        if review_len <= seq_length:
            zeros = list(np.zeros(seq_length - review_len, dtype=int))
            # Place zeros at the beginning for more effective learning
            # (RNNs prioritize later sequence elements)
            new_sequence = zeros + review
        # Handle long sequences: truncate to seq_length
        else:
            # Preserve the most recent part of the review (recency effect)
            new_sequence = review[-seq_length:]
        # Store the padded/truncated sequence
        features[i, :] = np.array(new_sequence)
    return features

# Determine optimal sequence length
# Common approach: Choose a value that preserves most reviews while limiting excessive padding
seq_length = 200  # This captures a significant portion of the reviews

# Apply padding/truncation
padded_features = pad_features(reviews_encoded, seq_length)

# Verify pad_features implementation with examples
short_idx = np.argmin([len(r) for r in reviews_encoded])
long_idx = np.argmax([len(r) for r in reviews_encoded])

To properly evaluate model performance and prevent overfitting, we divide the dataset into three distinct subsets: Train, Validation and Test sets

In [11]:
# Create data splits for training, validation, and testing
# Calculate split indices
total_samples = len(padded_features)
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

train_end = int(train_ratio * total_samples)
val_end = train_end + int(val_ratio * total_samples)

# Split features
X_train = padded_features[:train_end]
X_val = padded_features[train_end:val_end]
X_test = padded_features[val_end:]

# Split labels
y_train = encoded_labels[:train_end]
y_val = encoded_labels[train_end:val_end]
y_test = encoded_labels[val_end:]

PyTorch DataLoader Configuration
The final step in dataset preparation is creating PyTorch DataLoader objects, which provide:

Efficient Batch Processing: Loading data in mini-batches to optimize GPU memory usage
Shuffling: Randomizing the order of training samples to prevent order-dependent biases
Parallel Data Loading: Utilizing multiple worker processes for faster data retrieval
Automatic Conversion: Transforming NumPy arrays into PyTorch tensors
These DataLoaders serve as the interface between the preprocessed data and the neural network training loop.

In [12]:
# Convert NumPy arrays to PyTorch tensors
# Create TensorDatasets for each split
train_data = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_data = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_data = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

# Configure batch size
batch_size = 64

# Create DataLoader objects
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)  # No need to shuffle test data

# Display DataLoader configuration
#print("\nDataLoader Configuration:")
#print(f"  Batch size: {batch_size}")
#print(f"  Training batches: {len(train_loader)}")
#print(f"  Validation batches: {len(val_loader)}")
#print(f"  Test batches: {len(test_loader)}")

# Display a sample batch
#print("\nSample batch inspection:")
sample_x, sample_y = next(iter(train_loader))
#print(f"  Input batch shape: {sample_x.shape}")  # [batch_size, sequence_length]
#print(f"  Label batch shape: {sample_y.shape}")  # [batch_size]

RNN Model for Sentiment Classification
The first architecture we'll implement is a standard Recurrent Neural Network (RNN). This architecture includes:

Embedding Layer: Transforms integer tokens into dense vector representations
RNN Layer(s): Processes the sequence data, maintaining hidden state across timesteps
Dropout: Regularizes the network to prevent overfitting
Linear Layer: Maps the final hidden state to a sentiment prediction
Sigmoid Activation: Produces a probability output for binary classification
The RNN architecture serves as our baseline model, providing a foundation for comparison with more advanced recurrent architectures.

In [13]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim, hidden_dim, n_layers, drop_prob=0.5):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=drop_prob
        )

        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(hidden_dim, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        batch_size = x.size(0)

        # Convert token ids to embeddings
        embeds = self.embedding(x)
        # embeds: (batch_size, seq_len, embedding_dim)

        # Initialize hidden state h0 with zeros
        h0 = torch.zeros(
            self.n_layers,
            batch_size,
            self.hidden_dim,
            device=x.device
        )

        # Pass embeddings and initial hidden state to RNN
        rnn_out, _ = self.rnn(embeds, h0)
        # rnn_out: (batch_size, seq_len, hidden_dim)

        # Take output from last time step
        last_output = rnn_out[:, -1, :]
        # last_output: (batch_size, hidden_dim)

        out = self.dropout(last_output)
        out = self.fc(out)
        out = self.sigmoid(out)

        return out

In [14]:
# Instantiate the RNN model with hyperparameters
# Model hyperparameters
vocab_size = len(vocab_to_int) + 1  # +1 for padding token (0)
output_size = 1  # Binary classification
embedding_dim = 400  # Word embedding dimension
hidden_dim = 256  # Hidden state dimension
n_layers = 2  # Number of RNN layers

# Create model instance
rnn_model = SentimentRNN(vocab_size, output_size, embedding_dim, hidden_dim, n_layers)

In [15]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, output_size, embedding_dim, hidden_dim, n_layers, drop_prob=0.5):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=drop_prob
        )

        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(hidden_dim, output_size)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        batch_size = x.size(0)

        # Convert token ids to embeddings
        embeds = self.embedding(x)
        # embeds: (batch_size, seq_len, embedding_dim)

        # Initialize hidden state h0 and cell state c0 with zeros
        h0 = torch.zeros(
            self.n_layers,
            batch_size,
            self.hidden_dim,
            device=x.device
        )

        c0 = torch.zeros(
            self.n_layers,
            batch_size,
            self.hidden_dim,
            device=x.device
        )

        # Pass embeddings and initial states to LSTM
        lstm_out, _ = self.lstm(embeds, (h0, c0))
        # lstm_out: (batch_size, seq_len, hidden_dim)

        # Take output from last time step
        last_output = lstm_out[:, -1, :]
        # last_output: (batch_size, hidden_dim)

        out = self.dropout(last_output)
        out = self.fc(out)
        out = self.sigmoid(out)

        return out

In [16]:
# Instantiate the LSTM model with the same hyperparameters as the RNN for fair comparison
print("Initializing LSTM model architecture...")

# Create model instance (using same hyperparameters as RNN)
lstm_model = SentimentLSTM(vocab_size, output_size, embedding_dim, hidden_dim, n_layers)

# Display model architecture summary
print(lstm_model)

Initializing LSTM model architecture...
SentimentLSTM(
  (embedding): Embedding(39582, 400)
  (lstm): LSTM(400, 256, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


Training Configuration

In [17]:
# Set random seed for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Move models to device
rnn_model = rnn_model.to(device)
lstm_model = lstm_model.to(device)

# Training hyperparameters
batch_size = 64
learning_rate = 0.001
epochs = 5

# Loss function
criterion = nn.BCELoss()

# Optimizers - one for each model
rnn_optimizer = torch.optim.Adam(rnn_model.parameters(), lr=learning_rate)
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=learning_rate)



Training on: cpu


Model Training and Evaluation

In [18]:
def calculate_metrics(y_pred, y_true, threshold=0.5):
    """
    Calculate classification metrics from model predictions.

    Parameters:
    -----------
    y_pred : torch.Tensor or numpy.ndarray
        Predicted probabilities
    y_true : torch.Tensor or numpy.ndarray
        Ground truth labels
    threshold : float, default=0.5
        Threshold for binary classification

    Returns:
    --------
    dict
        Dictionary containing accuracy, precision, recall, and F1-score
    """
    # Convert tensors to numpy if needed
    if isinstance(y_pred, torch.Tensor):
        y_pred = y_pred.cpu().detach().numpy()
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.cpu().detach().numpy()

    # Convert probabilities to binary predictions using threshold
    y_pred_binary = (y_pred > threshold).astype(int)

    # Calculate metrics
    accuracy = np.mean(y_pred_binary == y_true)

    # Prevent division by zero
    tp = np.sum((y_pred_binary == 1) & (y_true == 1))
    fp = np.sum((y_pred_binary == 1) & (y_true == 0))
    fn = np.sum((y_pred_binary == 0) & (y_true == 1))

    # Calculate precision, recall, and F1-score
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

In [19]:
def validate(model, val_loader, criterion, device, model_type=None):
    """
    Validate model on validation set.
    Works for simplified RNN/LSTM models with internal hidden-state initialization.
    """
    model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).float()

            # Forward pass
            outputs = model(inputs)

            # Compute loss
            loss = criterion(outputs.squeeze(), labels)
            total_val_loss += loss.item()

            # Store predictions and labels
            all_preds.extend(outputs.squeeze().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    metrics = calculate_metrics(np.array(all_preds), np.array(all_labels))

    return avg_val_loss, metrics


def test_model(model, test_loader, criterion, device, model_type=None):
    """
    Test model on test set.
    Works for simplified RNN/LSTM models with internal hidden-state initialization.
    """
    model.eval()

    total_test_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).float()

            # Forward pass
            outputs = model(inputs)

            # Compute loss
            loss = criterion(outputs.squeeze(), labels)
            total_test_loss += loss.item()

            # Store predictions and labels
            all_preds.extend(outputs.squeeze().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_test_loss = total_test_loss / len(test_loader)
    metrics = calculate_metrics(np.array(all_preds), np.array(all_labels))

    return avg_test_loss, metrics

In [20]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    epochs,
    device,
    model_type,
    patience=3,
    min_delta=0.0
):
    """
    Train the model with validation and early stopping.
    Works for both simplified RNN and LSTM models
    where hidden states are initialized inside forward().
    """

    model.to(device)

    train_losses = []
    val_losses = []
    val_metrics = []

    best_val_loss = float('inf')
    patience_counter = 0

    start_time = time.time()

    print(f"\n=== Training {model_type.upper()} Model ===")

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        # -------------------- TRAIN --------------------
        model.train()
        total_train_loss = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).float()

            # Clear previous gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)

            # Compute loss
            loss = criterion(outputs.squeeze(), labels)

            # Backpropagation
            loss.backward()

            # Update weights
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # -------------------- VALIDATION --------------------
        val_loss, metrics = validate(model, val_loader, criterion, device, model_type)
        val_losses.append(val_loss)
        val_metrics.append(metrics)

        epoch_time = time.time() - epoch_start

        # -------------------- LOGGING --------------------
        print(f"Epoch {epoch}/{epochs} | Time: {epoch_time:.2f}s")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f} | Val Accuracy: {metrics['accuracy']:.4f}")
        print(
            f"  Val Precision: {metrics['precision']:.4f} | "
            f"Val Recall: {metrics['recall']:.4f} | "
            f"Val F1: {metrics['f1_score']:.4f}"
        )

        # -------------------- EARLY STOPPING --------------------
        if val_loss < best_val_loss - min_delta:
            print(f"  Validation loss improved from {best_val_loss:.4f} to {val_loss:.4f}")

            best_val_loss = val_loss
            patience_counter = 0

            torch.save(model.state_dict(), f"models/{model_type}_best.pt")
            print(f"  Saved best model to models/{model_type}_best.pt")

        else:
            patience_counter += 1
            print(f"  Validation loss did not improve. Patience: {patience_counter}/{patience}")



    total_time = time.time() - start_time
    print(f"\nTraining completed in {total_time:.2f} seconds")

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_metrics": val_metrics,
        "best_val_loss": best_val_loss,
        "training_time": total_time
    }

Model Training: RNN Model

In [21]:
import os

# Create the 'models' directory if it doesn't exist
if not os.path.exists('models'):
    os.makedirs('models')

# Train RNN model
print("\n" + "="*50)
print("BEGINNING RNN TRAINING")
print("="*50)
rnn_history = train_model(
    model=rnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=rnn_optimizer,
    criterion=criterion,
    epochs=epochs,
    device=device,
    model_type='rnn'
)


BEGINNING RNN TRAINING

=== Training RNN Model ===
Epoch 1/5 | Time: 429.09s
  Train Loss: 0.6909
  Val Loss: 0.6496 | Val Accuracy: 0.6092
  Val Precision: 0.6471 | Val Recall: 0.5176 | Val F1: 0.5752
  Validation loss improved from inf to 0.6496
  Saved best model to models/rnn_best.pt
Epoch 2/5 | Time: 168.28s
  Train Loss: 0.5767
  Val Loss: 0.6799 | Val Accuracy: 0.6052
  Val Precision: 0.6394 | Val Recall: 0.5216 | Val F1: 0.5745
  Validation loss did not improve. Patience: 1/3
Epoch 3/5 | Time: 305.93s
  Train Loss: 0.4702
  Val Loss: 0.6448 | Val Accuracy: 0.6593
  Val Precision: 0.6557 | Val Recall: 0.7020 | Val F1: 0.6780
  Validation loss improved from 0.6496 to 0.6448
  Saved best model to models/rnn_best.pt
Epoch 4/5 | Time: 243.76s
  Train Loss: 0.3267
  Val Loss: 0.8603 | Val Accuracy: 0.6333
  Val Precision: 0.6216 | Val Recall: 0.7216 | Val F1: 0.6679
  Validation loss did not improve. Patience: 1/3
Epoch 5/5 | Time: 217.90s
  Train Loss: 0.2229
  Val Loss: 0.8051 | V

Model Training: LSTM Model

In [22]:
# Train LSTM model
print("\n" + "="*50)
print("BEGINNING LSTM TRAINING")
print("="*50)
lstm_history = train_model(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=lstm_optimizer,
    criterion=criterion,
    epochs=epochs,
    device=device,
    model_type='lstm'
)


BEGINNING LSTM TRAINING

=== Training LSTM Model ===
Epoch 1/5 | Time: 812.09s
  Train Loss: 0.6414
  Val Loss: 0.5618 | Val Accuracy: 0.7255
  Val Precision: 0.7810 | Val Recall: 0.6431 | Val F1: 0.7054
  Validation loss improved from inf to 0.5618
  Saved best model to models/lstm_best.pt
Epoch 2/5 | Time: 185.45s
  Train Loss: 0.4868
  Val Loss: 0.5749 | Val Accuracy: 0.7355
  Val Precision: 0.7204 | Val Recall: 0.7882 | Val F1: 0.7528
  Validation loss did not improve. Patience: 1/3
Epoch 3/5 | Time: 184.36s
  Train Loss: 0.3050
  Val Loss: 0.6480 | Val Accuracy: 0.6914
  Val Precision: 0.7760 | Val Recall: 0.5569 | Val F1: 0.6484
  Validation loss did not improve. Patience: 2/3
Epoch 4/5 | Time: 183.84s
  Train Loss: 0.1755
  Val Loss: 0.8709 | Val Accuracy: 0.6794
  Val Precision: 0.7624 | Val Recall: 0.5412 | Val F1: 0.6330
  Validation loss did not improve. Patience: 3/3
Epoch 5/5 | Time: 184.46s
  Train Loss: 0.0695
  Val Loss: 1.0384 | Val Accuracy: 0.7174
  Val Precision: 0

Model Evaluation and Comparison

In [23]:
# Load best models
rnn_model.load_state_dict(torch.load('models/rnn_best.pt'))
lstm_model.load_state_dict(torch.load('models/lstm_best.pt'))

# Evaluate on test set
print("\n--- Evaluating Models on Test Set ---")

# Test RNN model
print("\nRNN Model Performance:")
rnn_test_loss, rnn_metrics = test_model(rnn_model, test_loader, criterion, device, 'rnn')
print(f"Test Loss: {rnn_test_loss:.4f}")
print(f"Test Accuracy: {rnn_metrics['accuracy']:.4f}")
print(f"Test Precision: {rnn_metrics['precision']:.4f}")
print(f"Test Recall: {rnn_metrics['recall']:.4f}")
print(f"Test F1-Score: {rnn_metrics['f1_score']:.4f}")

# Test LSTM model
print("\nLSTM Model Performance:")
lstm_test_loss, lstm_metrics = test_model(lstm_model, test_loader, criterion, device, 'lstm')
print(f"Test Loss: {lstm_test_loss:.4f}")
print(f"Test Accuracy: {lstm_metrics['accuracy']:.4f}")
print(f"Test Precision: {lstm_metrics['precision']:.4f}")
print(f"Test Recall: {lstm_metrics['recall']:.4f}")
print(f"Test F1-Score: {lstm_metrics['f1_score']:.4f}")


--- Evaluating Models on Test Set ---

RNN Model Performance:
Test Loss: 0.6621
Test Accuracy: 0.6527
Test Precision: 0.6201
Test Recall: 0.7178
Test F1-Score: 0.6654

LSTM Model Performance:
Test Loss: 0.5634
Test Accuracy: 0.7126
Test Precision: 0.7413
Test Recall: 0.6183
Test F1-Score: 0.6742


Model Inference with User Input

In [24]:
import torch
import re
import string

def predict_sentiment(model, review_text, device, vocab_to_int, sequence_length=200):
    """
    Predict sentiment of a review using a trained RNN/LSTM model.
    """

    model.eval()

    # -------------------- PREPROCESS TEXT --------------------
    review_text = review_text.lower()

    # Remove HTML tags
    review_text = re.sub(r'<.*?>', '', review_text)

    # Remove punctuation
    punctuation = string.punctuation
    review_text = ''.join(' ' if c in punctuation else c for c in review_text)

    # Tokenize
    words = review_text.split()

    # Convert words to integer ids
    word_indices = [
        vocab_to_int[word] if word in vocab_to_int else 0
        for word in words
    ]

    # -------------------- PAD / TRUNCATE --------------------
    if len(word_indices) > sequence_length:
        word_indices = word_indices[-sequence_length:]
    else:
        padding = [0] * (sequence_length - len(word_indices))
        word_indices = padding + word_indices

    # Convert to tensor: shape (1, seq_len)
    inputs = torch.tensor(word_indices, dtype=torch.long).unsqueeze(0).to(device)

    # -------------------- INFERENCE --------------------
    with torch.no_grad():
        output = model(inputs)

    prob = output.item()

    sentiment = "Positive" if prob >= 0.5 else "Negative"
    confidence = prob if prob >= 0.5 else (1 - prob)

    return sentiment, confidence * 100

In [26]:
def analyze_review(review_text):
    """
    Analyze a review using both trained RNN and LSTM models.
    """

    print("\n" + "=" * 60)

    if len(review_text) > 100:
        print(f"Analyzing review: {review_text[:100]}...")
    else:
        print(f"Analyzing review: {review_text}")

    print("=" * 60)

    # Predict using both models
    rnn_sentiment, rnn_confidence = predict_sentiment(
        rnn_model, review_text, device, vocab_to_int
    )

    lstm_sentiment, lstm_confidence = predict_sentiment(
        lstm_model, review_text, device, vocab_to_int
    )

    # Show results
    print("\nResults:")
    print(f"RNN Model  : {rnn_sentiment} (Confidence: {rnn_confidence:.2f}%)")
    print(f"LSTM Model : {lstm_sentiment} (Confidence: {lstm_confidence:.2f}%)")

    # Compare predictions
    if rnn_sentiment == lstm_sentiment:
        print(
            f"\nBoth models agree: This review expresses "
            f"{rnn_sentiment.lower()} sentiment."
        )
    else:
        print("\nModels disagree on sentiment classification.")
        print(f"RNN predicts  {rnn_sentiment.lower()} ({rnn_confidence:.2f}%)")
        print(f"LSTM predicts {lstm_sentiment.lower()} ({lstm_confidence:.2f}%)")
        print("You may want to trust the higher-confidence prediction.")

In [27]:
# Test with a few example reviews
example_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged from start to finish.",
    "I was very disappointed with this film. The characters were poorly developed and the story made no sense at all.",
    "It was an okay movie. Some parts were good but others were boring. Not great, not terrible.",
    "The special effects were amazing but everything else about this movie was terrible."
]

# Analyze each example review
for i, review in enumerate(example_reviews):
    print(f"\nExample {i+1}:")
    analyze_review(review)


Example 1:

Analyzing review: This movie was absolutely fantastic! The acting was superb and the plot kept me engaged from start t...

Results:
RNN Model  : Positive (Confidence: 55.18%)
LSTM Model : Positive (Confidence: 50.28%)

Both models agree: This review expresses positive sentiment.

Example 2:

Analyzing review: I was very disappointed with this film. The characters were poorly developed and the story made no s...

Results:
RNN Model  : Negative (Confidence: 90.10%)
LSTM Model : Negative (Confidence: 65.69%)

Both models agree: This review expresses negative sentiment.

Example 3:

Analyzing review: It was an okay movie. Some parts were good but others were boring. Not great, not terrible.

Results:
RNN Model  : Negative (Confidence: 93.02%)
LSTM Model : Negative (Confidence: 55.08%)

Both models agree: This review expresses negative sentiment.

Example 4:

Analyzing review: The special effects were amazing but everything else about this movie was terrible.

Results:
RNN Mode

In [ ]:
# Interactive input for user reviews
print("\n" + "="*60)
print("Enter your own movie review to analyze (or type 'exit' to quit):")
print("="*60)

while True:
    user_input = input("\nEnter a movie review: ")
    if user_input.lower() == 'exit':
        print("Exiting sentiment analysis.")
        break
    if not user_input.strip():
        print("Please enter a review or type 'exit'.")
        continue

    analyze_review(user_input)


Enter your own movie review to analyze (or type 'exit' to quit):

Enter a movie review: Movie excelled in music and acting. But the direction and screenplay were disappointing

Analyzing review: Movie excelled in music and acting. But the direction and screenplay were disappointing

Results:
RNN Model  : Negative (Confidence: 95.11%)
LSTM Model : Negative (Confidence: 90.93%)

Both models agree: This review expresses negative sentiment.

Enter a movie review: exit
Exiting sentiment analysis.
